# 05 — Game 28th Birthday · App perf

Pulls Omni app/B2B/B2C sales from the DWH and auto-pastes into `agg_app_manual!O1`.

## 1. Libraries

In [ ]:
import oracledb
import sys

oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
import cx_Oracle

import pandas as pd
import os
import time
import subprocess
import webbrowser
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

## 2. Oracle client init

In [ ]:
instant_client_path = r"D:\oracle\instantclient_19_29"
cx_Oracle.init_oracle_client(lib_dir=instant_client_path)
print("Oracle client initialized.")

## 3. Config & query

Change `start_date` / `end_date` per run.

In [ ]:
ip           = os.getenv("DWH_HOST")
port         = os.getenv("DWH_PORT")
service_name = os.getenv("DWH_NAME")
username     = os.getenv("DWH_USER")
password     = os.getenv("DWH_PASSWORD")

start_date = "16-JUN-2026"
end_date   = "16-JUN-2026"
# yesterday: start_date = end_date = (datetime.now() - timedelta(days=1)).strftime("%d-%b-%Y").upper()

sheet_url = (
    "https://docs.google.com/spreadsheets/d/"
    "1gUGVcOBaMjERvTr56IRYbdgBR35Jd1nWTki9PH-wkjU/edit"
    "?gid=73230870#gid=73230870&range=O1"
)

sql_script = f"""
SELECT
    sale_date AS ds,
    dimension,
    net_sales,
    ord_cnt
FROM
    crv_data.LOUTRUONG_SUPPLIER_PERF_DI
WHERE
    1 = 1
    AND (sale_date BETWEEN '{start_date}' AND '{end_date}')
    AND LOWER(supplier_code)    IN ('00_all_omni')
    AND LOWER(dimension_group)  IN ('channel', 'customer_type')
    AND LOWER(dimension)        IN ('app', 'b2c')
ORDER BY
    sale_date ASC,
    dimension ASC
"""
print(sql_script)

## 4. Connect

In [ ]:
dsn_tns    = cx_Oracle.makedsn(ip, port, service_name=service_name)
connection = cx_Oracle.connect(user=username, password=password, dsn=dsn_tns)
cursor     = connection.cursor()
print("Connected.")

## 5. Query

In [ ]:
cursor.execute(sql_script)
columns = [c[0] for c in cursor.description]
df = pd.DataFrame(cursor.fetchall(), columns=columns)
print(f"Rows returned: {len(df)}")

## 6. Preview

In [ ]:
print(df.head(20).to_string(index=False))
print()
print(df.dtypes)

## 7. Format for paste

Date -> `YYYY-MM-DD` text; numbers stay numeric.

In [ ]:
df_out = df.copy()
date_col = df_out.columns[0]
df_out[date_col] = pd.to_datetime(df_out[date_col]).dt.strftime("%Y-%m-%d")
df_out

## 8. Copy -> Chrome -> paste

Installs `pyautogui` if missing. Be logged into Google; keep hands off the mouse/keyboard
during the wait. If the paste misses, raise `PAGE_LOAD_WAIT` or press Ctrl+V yourself.

In [ ]:
PAGE_LOAD_WAIT = 10   # seconds for the sheet to load before pasting

try:
    import pyautogui
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyautogui"])
    import pyautogui
pyautogui.FAILSAFE = True

df_out.to_clipboard(index=False)
# df_out.to_csv("05_game28thbirthday_app_perf.csv", index=False)   # optional backup

chrome_paths = [
    r"C:\Program Files\Google\Chrome\Application\chrome.exe",
    r"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe",
    os.path.expandvars(r"%LOCALAPPDATA%\Google\Chrome\Application\chrome.exe"),
]
chrome = next((p for p in chrome_paths if os.path.exists(p)), None)
subprocess.Popen([chrome, sheet_url]) if chrome else webbrowser.open(sheet_url)

print(f"Opening the sheet... waiting {PAGE_LOAD_WAIT}s (hands off the keyboard/mouse).")
time.sleep(PAGE_LOAD_WAIT)
pyautogui.hotkey("ctrl", "v")
print("Pasted into O1.")

## 9. Close

In [ ]:
cursor.close()
connection.close()
print("Connection closed.")